<a href="https://colab.research.google.com/github/eshan14git/football-qa-nlp/blob/disath-dev/notebooks/football_qa_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Interactive Football Question Intent Classifier

## Notebook Overview

This notebook provides an interactive interface for testing the two intent-classification
models developed for the Football Natural Language Question Answering project:

1. Word-and-character TF-IDF Logistic Regression
2. Word-level one-dimensional Convolutional Neural Network

Users can enter a football question and test either model individually or compare both models.
The interface displays each model's predicted intent and confidence score.

This demonstration currently performs intent classification. It identifies the type of
football information requested but does not yet retrieve the final answer from the underlying
football datasets.

## Connect Drive and import libraries

In [1]:
from google.colab import drive
drive.mount("/content/drive")

import os
import json
import pickle
import warnings

import numpy as np
import pandas as pd
import joblib
import tensorflow as tf

from tensorflow.keras.preprocessing.sequence import pad_sequences

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning
)

print("TensorFlow version:", tf.__version__)
print("Demo runtime ready.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
TensorFlow version: 2.20.0
Demo runtime ready.


## Locate the Saved Model Artifacts

The demonstration loads the finalized model and preprocessing artifacts created by the ML and
DL notebooks. All required files are validated before either model is used.

## Define and validate artifact paths

In [2]:
MODEL_DIRECTORY = (
    "/content/drive/MyDrive/NLP/models"
)

ML_VECTORIZER_PATH = os.path.join(
    MODEL_DIRECTORY,
    "disa_word_character_tfidf_vectorizer.joblib"
)

ML_MODEL_PATH = os.path.join(
    MODEL_DIRECTORY,
    "disa_logistic_regression_intent_model.joblib"
)

CNN_MODEL_PATH = os.path.join(
    MODEL_DIRECTORY,
    "disa_cnn_intent_model.keras"
)

CNN_TOKENIZER_PATH = os.path.join(
    MODEL_DIRECTORY,
    "disa_cnn_tokenizer.pkl"
)

CNN_LABEL_ENCODER_PATH = os.path.join(
    MODEL_DIRECTORY,
    "disa_cnn_label_encoder.joblib"
)

CNN_CONFIG_PATH = os.path.join(
    MODEL_DIRECTORY,
    "disa_cnn_config.json"
)

artifact_paths = {
    "ML vectorizer": ML_VECTORIZER_PATH,
    "ML model": ML_MODEL_PATH,
    "CNN model": CNN_MODEL_PATH,
    "CNN tokenizer": CNN_TOKENIZER_PATH,
    "CNN label encoder": CNN_LABEL_ENCODER_PATH,
    "CNN configuration": CNN_CONFIG_PATH
}

print("MODEL ARTIFACT VALIDATION")
print("-" * 70)

missing_artifacts = []

for artifact_name, artifact_path in artifact_paths.items():
    artifact_exists = os.path.exists(
        artifact_path
    )

    status = (
        "FOUND"
        if artifact_exists
        else "MISSING"
    )

    print(
        f"{artifact_name:<22} "
        f"{status:<8} "
        f"{artifact_path}"
    )

    if not artifact_exists:
        missing_artifacts.append(
            artifact_name
        )

if missing_artifacts:
    raise FileNotFoundError(
        "Missing required artifacts: "
        + ", ".join(missing_artifacts)
    )

print("\nAll required model artifacts were found.")

MODEL ARTIFACT VALIDATION
----------------------------------------------------------------------
ML vectorizer          FOUND    /content/drive/MyDrive/NLP/models/disa_word_character_tfidf_vectorizer.joblib
ML model               FOUND    /content/drive/MyDrive/NLP/models/disa_logistic_regression_intent_model.joblib
CNN model              FOUND    /content/drive/MyDrive/NLP/models/disa_cnn_intent_model.keras
CNN tokenizer          FOUND    /content/drive/MyDrive/NLP/models/disa_cnn_tokenizer.pkl
CNN label encoder      FOUND    /content/drive/MyDrive/NLP/models/disa_cnn_label_encoder.joblib
CNN configuration      FOUND    /content/drive/MyDrive/NLP/models/disa_cnn_config.json

All required model artifacts were found.


## Load the ML and CNN Models

Each trained model is loaded together with its required preprocessing components. Their intent
classes and CNN configuration are inspected to confirm compatibility before prediction
functions are created.

## Load all artifacts

In [3]:
ml_vectorizer = joblib.load(
    ML_VECTORIZER_PATH
)

ml_model = joblib.load(
    ML_MODEL_PATH
)

cnn_model = tf.keras.models.load_model(
    CNN_MODEL_PATH
)

with open(
    CNN_TOKENIZER_PATH,
    "rb"
) as tokenizer_file:
    cnn_tokenizer = pickle.load(
        tokenizer_file
    )

cnn_label_encoder = joblib.load(
    CNN_LABEL_ENCODER_PATH
)

with open(
    CNN_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as configuration_file:
    cnn_configuration = json.load(
        configuration_file
    )

print("MODEL ARTIFACTS LOADED")
print("-" * 60)
print(
    "ML vectorizer type:",
    type(ml_vectorizer).__name__
)
print(
    "ML model type:",
    type(ml_model).__name__
)
print(
    "CNN model name:",
    cnn_model.name
)
print(
    "CNN input shape:",
    cnn_model.input_shape
)
print(
    "CNN output shape:",
    cnn_model.output_shape
)

MODEL ARTIFACTS LOADED
------------------------------------------------------------
ML vectorizer type: FeatureUnion
ML model type: LogisticRegression
CNN model name: football_intent_cnn
CNN input shape: (None, 25)
CNN output shape: (None, 5)


## Verify intent classes

In [4]:
ml_intent_classes = list(
    ml_model.classes_
)

cnn_intent_classes = list(
    cnn_label_encoder.classes_
)

configured_cnn_classes = (
    cnn_configuration["intent_classes"]
)

print("INTENT-CLASS COMPATIBILITY")
print("-" * 60)

print("\nML classes:")
for class_index, intent in enumerate(
    ml_intent_classes
):
    print(f"{class_index}: {intent}")

print("\nCNN classes:")
for class_index, intent in enumerate(
    cnn_intent_classes
):
    print(f"{class_index}: {intent}")

print(
    "\nConfigured maximum sequence length:",
    cnn_configuration["max_sequence_length"]
)

assert ml_intent_classes == cnn_intent_classes
assert cnn_intent_classes == configured_cnn_classes

assert cnn_model.output_shape[-1] == len(
    cnn_intent_classes
)

print(
    "\nBoth models use the same intent classes "
    "in the same order."
)

INTENT-CLASS COMPATIBILITY
------------------------------------------------------------

ML classes:
0: match_score
1: match_scorers
2: match_winner
3: player_match_goal_count
4: player_match_scoring_minutes

CNN classes:
0: match_score
1: match_scorers
2: match_winner
3: player_match_goal_count
4: player_match_scoring_minutes

Configured maximum sequence length: 25

Both models use the same intent classes in the same order.


## Define Model-Specific Text Preparation

The Logistic Regression model and CNN use different preprocessing procedures. The ML model
uses lowercase text with restricted punctuation, matching its training notebook. The CNN uses
minimal whitespace normalization before tokenization.

Each new question must be transformed using the preprocessing method associated with the
selected model.

## Define both preprocessing function

In [5]:
import re

def clean_ml_question_text(text):
    """
    Apply the same text cleaning used by the ML notebook.
    """
    text = str(text).lower()

    # Normalize apostrophes
    text = text.replace("’", "'")

    # Remove punctuation while retaining letters,
    # numbers, apostrophes and hyphens
    text = re.sub(
        r"[^a-z0-9\s'-]",
        " ",
        text
    )

    # Replace repeated whitespace with one space
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


def prepare_cnn_text(text):
    """
    Apply the same minimal normalization used by the CNN notebook.
    """
    if pd.isna(text):
        return ""

    normalized_text = str(text).strip()

    normalized_text = " ".join(
        normalized_text.split()
    )

    return normalized_text

## Verify preprocessing

In [6]:
preprocessing_example = (
    "Wht was the final scor in Brazil versus Chile?"
)

ml_prepared_example = clean_ml_question_text(
    preprocessing_example
)

cnn_prepared_example = prepare_cnn_text(
    preprocessing_example
)

print("PREPROCESSING VERIFICATION")
print("-" * 60)
print("Original:")
print(preprocessing_example)

print("\nML-prepared:")
print(ml_prepared_example)

print("\nCNN-prepared:")
print(cnn_prepared_example)

assert ml_prepared_example == (
    "wht was the final scor in brazil versus chile"
)

assert cnn_prepared_example == preprocessing_example

print("\nBoth preprocessing functions work correctly.")

PREPROCESSING VERIFICATION
------------------------------------------------------------
Original:
Wht was the final scor in Brazil versus Chile?

ML-prepared:
wht was the final scor in brazil versus chile

CNN-prepared:
Wht was the final scor in Brazil versus Chile?

Both preprocessing functions work correctly.


## Define Prediction Functions

Separate prediction functions apply the correct preprocessing and model-specific
transformations. Each function returns the predicted intent and the model's maximum class
probability as its confidence score.

## Logistic Regression prediction function

In [7]:
def predict_ml_intent(question):
    """
    Predict one question's intent using Logistic Regression.
    """
    if not isinstance(question, str) or not question.strip():
        raise ValueError(
            "The question must contain non-empty text."
        )

    cleaned_question = clean_ml_question_text(
        question
    )

    question_features = ml_vectorizer.transform(
        [cleaned_question]
    )

    probability_vector = ml_model.predict_proba(
        question_features
    )[0]

    predicted_class_index = int(
        np.argmax(probability_vector)
    )

    predicted_intent = ml_model.classes_[
        predicted_class_index
    ]

    confidence = float(
        probability_vector[
            predicted_class_index
        ]
    )

    return {
        "model": "Logistic Regression",
        "predicted_intent": predicted_intent,
        "confidence": confidence
    }

## CNN prediction function

In [8]:
def predict_cnn_intent(question):
    """
    Predict one question's intent using the CNN.
    """
    if not isinstance(question, str) or not question.strip():
        raise ValueError(
            "The question must contain non-empty text."
        )

    prepared_question = prepare_cnn_text(
        question
    )

    question_sequence = (
        cnn_tokenizer.texts_to_sequences(
            [prepared_question]
        )
    )

    padded_question = pad_sequences(
        question_sequence,
        maxlen=cnn_configuration[
            "max_sequence_length"
        ],
        padding="post",
        truncating="post"
    )

    probability_vector = cnn_model.predict(
        padded_question,
        verbose=0
    )[0]

    predicted_class_index = int(
        np.argmax(probability_vector)
    )

    predicted_intent = (
        cnn_label_encoder.inverse_transform(
            [predicted_class_index]
        )[0]
    )

    confidence = float(
        probability_vector[
            predicted_class_index
        ]
    )

    return {
        "model": "1D CNN",
        "predicted_intent": predicted_intent,
        "confidence": confidence
    }

## Test both models

In [9]:
test_questions = [
    "Who won between Brazil and Germany?",
    "What was the final score in France versus Argentina?",
    "Who scored in the Spain and Italy match?",
    "How many goals did Lionel Messi score?",
    "At what minutes did Cristiano Ronaldo score?"
]

comparison_results = []

for question in test_questions:
    ml_result = predict_ml_intent(
        question
    )

    cnn_result = predict_cnn_intent(
        question
    )

    comparison_results.append(
        {
            "question": question,
            "ML prediction": (
                ml_result["predicted_intent"]
            ),
            "ML confidence": (
                ml_result["confidence"]
            ),
            "CNN prediction": (
                cnn_result["predicted_intent"]
            ),
            "CNN confidence": (
                cnn_result["confidence"]
            ),
            "models_agree": (
                ml_result["predicted_intent"]
                == cnn_result["predicted_intent"]
            )
        }
    )

comparison_results_df = pd.DataFrame(
    comparison_results
)

print("MODEL PREDICTION VERIFICATION")
print("-" * 70)

display(
    comparison_results_df.style.format({
        "ML confidence": "{:.4f}",
        "CNN confidence": "{:.4f}"
    })
)

MODEL PREDICTION VERIFICATION
----------------------------------------------------------------------


,question,ML prediction,ML confidence,CNN prediction,CNN confidence,models_agree
0,Who won between Brazil and Germany?,match_winner,0.4670,match_winner,0.9815,True
1,What was the final score in France versus Argentina?,match_score,0.9319,match_score,1.0000,True
2,Who scored in the Spain and Italy match?,match_scorers,0.8253,match_scorers,1.0000,True
3,How many goals did Lionel Messi score?,player_match_goal_count,0.9025,player_match_goal_count,1.0000,True
4,At what minutes did Cristiano Ronaldo score?,player_match_scoring_minutes,0.8915,player_match_scoring_minutes,1.0000,True


## Interactive Gradio Interface

The interface allows a user to enter a football question and select Logistic Regression, CNN,
or both models. It displays the predicted intent and confidence for the selected model or
models.

The interface performs intent classification only and does not yet retrieve the factual answer
from the football datasets.

## Install and import Gradio

In [10]:
!pip install -q gradio

import gradio as gr

print("Gradio version:", gr.__version__)

Gradio version: 6.20.0


## Interface prediction handler

In [11]:
def classify_football_question(
    question,
    model_choice
):
    """
    Classify a football question using the selected model.
    """
    if not isinstance(question, str) or not question.strip():
        return pd.DataFrame(
            [
                {
                    "Model": "No model executed",
                    "Predicted Intent": (
                        "Please enter a football question."
                    ),
                    "Confidence": 0.0
                }
            ]
        )

    prediction_results = []

    if model_choice in [
        "Logistic Regression",
        "Compare Both"
    ]:
        ml_result = predict_ml_intent(
            question
        )

        prediction_results.append(
            {
                "Model": ml_result["model"],
                "Predicted Intent": (
                    ml_result["predicted_intent"]
                ),
                "Confidence": (
                    ml_result["confidence"]
                )
            }
        )

    if model_choice in [
        "1D CNN",
        "Compare Both"
    ]:
        cnn_result = predict_cnn_intent(
            question
        )

        prediction_results.append(
            {
                "Model": cnn_result["model"],
                "Predicted Intent": (
                    cnn_result["predicted_intent"]
                ),
                "Confidence": (
                    cnn_result["confidence"]
                )
            }
        )

    results_df = pd.DataFrame(
        prediction_results
    )

    results_df["Confidence"] = (
        results_df["Confidence"]
        .round(4)
    )

    return results_df

## Build and launch the interface

In [ ]:
benchmark_results_df = pd.DataFrame(
    {
        "Model": [
            "Logistic Regression",
            "1D CNN"
        ],
        "Generated-Test Accuracy": [
            "100%",
            "100%"
        ],
        "Generated-Test Macro F1": [
            "1.0000",
            "1.0000"
        ],
        "Manual Accuracy": [
            "74%",
            "66%"
        ],
        "Manual Macro F1": [
            "0.7381",
            "0.6571"
        ]
    }
)


with gr.Blocks(
    title="Football Question Intent Classifier"
) as demo:

    gr.Markdown(
        """
        # Football Question Intent Classifier

        Enter a football question and choose which trained model to test.

        The system supports five intents:

        - `match_winner`
        - `match_score`
        - `match_scorers`
        - `player_match_goal_count`
        - `player_match_scoring_minutes`

        **Important:** Confidence is the probability assigned to the
        current prediction. It is not the model's overall accuracy.

        This demonstration classifies the question's intent. It does
        not yet retrieve the factual football answer.
        """
    )

    gr.Markdown(
        "## Official Evaluation Results"
    )

    gr.Dataframe(
        value=benchmark_results_df,
        headers=[
            "Model",
            "Generated-Test Accuracy",
            "Generated-Test Macro F1",
            "Manual Accuracy",
            "Manual Macro F1"
        ],
        label="Model Performance",
        interactive=False
    )

    gr.Markdown(
        """
        Both models achieved 100% on generated questions. On the
        independent manually written set, Logistic Regression achieved
        74% accuracy, while the CNN achieved 66%. The CNN may still
        display higher confidence because its probabilities are
        overconfident on unfamiliar wording.
        """
    )

    gr.Markdown(
        "## Test a Football Question"
    )

    question_input = gr.Textbox(
        lines=3,
        label="Football Question",
        placeholder=(
            "Example: Who scored in the Spain and Italy match?"
        )
    )

    model_selector = gr.Radio(
        choices=[
            "Logistic Regression",
            "1D CNN",
            "Compare Both"
        ],
        value="Compare Both",
        label="Select Model"
    )

    classify_button = gr.Button(
        "Classify Question",
        variant="primary"
    )

    prediction_output = gr.Dataframe(
        headers=[
            "Model",
            "Predicted Intent",
            "Confidence"
        ],
        datatype=[
            "str",
            "str",
            "number"
        ],
        label="Prediction Results",
        interactive=False
    )

    classify_button.click(
        fn=classify_football_question,
        inputs=[
            question_input,
            model_selector
        ],
        outputs=prediction_output
    )

    question_input.submit(
        fn=classify_football_question,
        inputs=[
            question_input,
            model_selector
        ],
        outputs=prediction_output
    )

    gr.Examples(
        examples=[
            [
                "Who won between Brazil and Germany?",
                "Compare Both"
            ],
            [
                "What was the final score in France versus Argentina?",
                "Compare Both"
            ],
            [
                "Who scored in the Spain and Italy match?",
                "Compare Both"
            ],
            [
                "How many goals did Lionel Messi score?",
                "Compare Both"
            ],
            [
                "At what minutes did Cristiano Ronaldo score?",
                "Compare Both"
            ]
        ],
        inputs=[
            question_input,
            model_selector
        ]
    )

demo.launch(
    share=True,
    debug=False
)